# Project Analysis: Netflix Content Popularity

**Course:** CMSE 202  
**Project Update Notebook**

**Group Members:**  
- Jaehan Kim — kimjaeha@msu.edu  
- Akshita Mylavarapu — mylavar2@msu.edu  
- Taliah Blom — blomtali@msu.edu  
- Ahmed Chishti — chishti2@msu.edu  

## Main Research Question

Can Netflix content features such as release year, rating, duration, genre, country, and content type help explain or predict popularity?

## Popularity Definition

In this project, popularity will mainly be measured using available popularity-related variables from the datasets, especially TMDB popularity, IMDb votes, IMDb score, and TMDB score.

## Dataset Sources

1. `Netflix Dataset.csv`  
   Source: https://www.kaggle.com/datasets/rohitgrewal/netflix-data

2. `netflix_titles.csv`  
   Source: https://www.kaggle.com/datasets/shivamb/netflix-shows

3. `credits.csv` and `titles.csv`  
   Source: https://www.kaggle.com/datasets/victorsoeiro/netflix-tv-shows-and-movies



---
# 1. Import packages

These packages are used for data organization, cleaning, and basic analysis.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

---
# 2. Load the datasets


In [2]:
# File names
rohit_file = "data/Netflix Dataset.csv"
shivamb_file = "data/netflix_titles.csv"
credits_file = "data/credits.csv"
titles_file = "data/titles.csv"

# required_files = [rohit_file, shivamb_file, credits_file, titles_file]

In [3]:
# Load datasets
#credits_raw = pd.read_csv(credits_file)
titles_raw = pd.read_csv(titles_file)

#print("Credits dataset shape:", credits_raw.shape)
print("Titles dataset shape:", titles_raw.shape)

display(titles_raw.head())

Titles dataset shape: (5850, 15)


,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


---
# 3. Check and understand the data

Before cleaning, checking the columns, data types, missing values, and basic summary statistics.


In [4]:
print("Column names:")
print(titles_raw.columns.tolist())

Column names:
['id', 'title', 'type', 'description', 'release_year', 'age_certification', 'runtime', 'genres', 'production_countries', 'seasons', 'imdb_id', 'imdb_score', 'imdb_votes', 'tmdb_popularity', 'tmdb_score']


In [5]:
titles_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5850 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    5850 non-null   object 
 1   title                 5849 non-null   object 
 2   type                  5850 non-null   object 
 3   description           5832 non-null   object 
 4   release_year          5850 non-null   int64  
 5   age_certification     3231 non-null   object 
 6   runtime               5850 non-null   int64  
 7   genres                5850 non-null   object 
 8   production_countries  5850 non-null   object 
 9   seasons               2106 non-null   float64
 10  imdb_id               5447 non-null   object 
 11  imdb_score            5368 non-null   float64
 12  imdb_votes            5352 non-null   float64
 13  tmdb_popularity       5759 non-null   float64
 14  tmdb_score            5539 non-null   float64
dtypes: float64(5), int64(

In [6]:
missing_values = titles_raw.isna().sum().sort_values(ascending=False)
display(missing_values)
display(titles_raw.describe())

seasons                 3744
age_certification       2619
imdb_votes               498
imdb_score               482
imdb_id                  403
tmdb_score               311
tmdb_popularity           91
description               18
title                      1
id                         0
type                       0
runtime                    0
release_year               0
genres                     0
production_countries       0
dtype: int64

,release_year,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
count,5850.000000,5850.000000,2106.000000,5368.000000,5.352000e+03,5759.000000,5539.000000
mean,2016.417094,76.888889,2.162868,6.510861,2.343938e+04,22.637925,6.829175
std,6.937726,39.002509,2.689041,1.163826,9.582047e+04,81.680263,1.170391
min,1945.000000,0.000000,1.000000,1.500000,5.000000e+00,0.009442,0.500000
25%,2016.000000,44.000000,1.000000,5.800000,5.167500e+02,2.728500,6.100000
50%,2018.000000,83.000000,1.000000,6.600000,2.233500e+03,6.821000,6.900000
75%,2020.000000,104.000000,2.000000,7.300000,9.494000e+03,16.590000,7.537500
max,2022.000000,240.000000,42.000000,9.600000,2.294231e+06,2274.044000,10.000000


# 4. Clean dataset

Columns needed from title.csv:

- Title
- type
- release_year
- runtime
- genres
- production_countries
- imdb_score
- imdb_votes
- tmdb_popularity
- tmdb_score

In [7]:
# Select only the columns we need
selected_columns = [
    "title",
    "type",
    "release_year",
    "runtime",
    "genres",
    "production_countries",
    "imdb_score",
    "imdb_votes",
    "tmdb_popularity",
    "tmdb_score"
]

titles_clean = titles_raw[selected_columns].copy()

display(titles_clean.head())


,title,type,release_year,runtime,genres,production_countries,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,Five Came Back: The Reference Films,SHOW,1945,51,['documentation'],['US'],NaN,NaN,0.600,NaN
1,Taxi Driver,MOVIE,1976,114,"['drama', 'crime']",['US'],8.2,808582.0,40.965,8.179
2,Deliverance,MOVIE,1972,109,"['drama', 'action', 'thriller', 'european']",['US'],7.7,107673.0,10.010,7.300
3,Monty Python and the Holy Grail,MOVIE,1975,91,"['fantasy', 'action', 'comedy']",['GB'],8.2,534486.0,15.461,7.811
4,The Dirty Dozen,MOVIE,1967,150,"['war', 'action']","['GB', 'US']",7.7,72662.0,20.398,7.600


In [8]:
# Removing duplicate rows
titles_clean = titles_clean.drop_duplicates()

# Dropping "[", "]", and "'" characters
titles_clean["genres"] = (
    titles_clean["genres"]
    .astype(str)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
)

titles_clean["production_countries"] = (
    titles_clean["production_countries"]
    .astype(str)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("'", "", regex=False)
)

display(titles_clean.head())

,title,type,release_year,runtime,genres,production_countries,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,Five Came Back: The Reference Films,SHOW,1945,51,documentation,US,NaN,NaN,0.600,NaN
1,Taxi Driver,MOVIE,1976,114,"drama, crime",US,8.2,808582.0,40.965,8.179
2,Deliverance,MOVIE,1972,109,"drama, action, thriller, european",US,7.7,107673.0,10.010,7.300
3,Monty Python and the Holy Grail,MOVIE,1975,91,"fantasy, action, comedy",GB,8.2,534486.0,15.461,7.811
4,The Dirty Dozen,MOVIE,1967,150,"war, action","GB, US",7.7,72662.0,20.398,7.600


In [9]:
# Convert number columns to numeric
number_columns = [
    "release_year",
    "runtime",
    "imdb_score",
    "imdb_votes",
    "tmdb_popularity",
    "tmdb_score"
]

for col in number_columns:
    titles_clean[col] = pd.to_numeric(titles_clean[col], errors="coerce")

# Removing rows that are missing important values
titles_clean = titles_clean.dropna(
    subset=[
        "title",
        "type",
        "release_year",
        "runtime",
        "tmdb_popularity",
        "imdb_score",
        "imdb_votes",
        "tmdb_score"
    ]
)

In [10]:
# Checking the resulted dataset
print("Cleaned titles dataset shape:", titles_clean.shape)
print("Cleaned titles dataset columns:")
print(titles_clean.columns.tolist())

display(titles_clean.head())
display(titles_clean.describe())

Cleaned titles dataset shape: (5131, 10)
Cleaned titles dataset columns:
['title', 'type', 'release_year', 'runtime', 'genres', 'production_countries', 'imdb_score', 'imdb_votes', 'tmdb_popularity', 'tmdb_score']


,title,type,release_year,runtime,genres,production_countries,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1,Taxi Driver,MOVIE,1976,114,"drama, crime",US,8.2,808582.0,40.965,8.179
2,Deliverance,MOVIE,1972,109,"drama, action, thriller, european",US,7.7,107673.0,10.010,7.300
3,Monty Python and the Holy Grail,MOVIE,1975,91,"fantasy, action, comedy",GB,8.2,534486.0,15.461,7.811
4,The Dirty Dozen,MOVIE,1967,150,"war, action","GB, US",7.7,72662.0,20.398,7.600
5,Monty Python's Flying Circus,SHOW,1969,30,"comedy, european",GB,8.8,73424.0,17.617,8.306


,release_year,runtime,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
count,5131.000000,5131.000000,5131.000000,5.131000e+03,5131.000000,5131.000000
mean,2016.288053,78.840577,6.516079,2.434477e+04,24.097710,6.825211
std,7.035558,38.615331,1.152650,9.754941e+04,85.280431,1.146735
min,1954.000000,0.000000,1.600000,5.000000e+00,0.600000,1.000000
25%,2016.000000,46.000000,5.800000,6.120000e+02,3.140500,6.100000
50%,2018.000000,86.000000,6.600000,2.474000e+03,7.479000,6.900000
75%,2020.000000,105.000000,7.300000,1.028300e+04,17.606500,7.500000
max,2022.000000,225.000000,9.500000,2.294231e+06,2274.044000,10.000000


In [11]:
# Save the cleaned dataset as a new CSV file
titles_clean.to_csv("data/cleaned_titles.csv", index=False)